Importacion de librerias para manejo del sistema

In [ ]:
import sys
import os

#Indicamos el workspace principal (dos niveles atrás)
sys.path.append(os.path.abspath('../../'))


In [2]:
#Manipulación y Análisis de Datos
import numpy as np
import pandas as pd
import pyreadstat
import plotly.express as px

#Funciones propias estadisticas
from src.estadisticas_pisa import consolidar_puntuaciones_pisa


# Carga de datos y exploracion

Para la lectura de datos, en vez de el metodo de pandas *read_sas* emplearemos [*pyreadstat*](https://github.com/Roche/pyreadstat/tree/master#reading-files) que presenta una mejor API para el manjo de datos SAS obteniendo el dataframe y sus metadatos

Realizamos la lectura de los metadatos, para obtener el nombre e informacion de las columnas 

In [3]:
#Realizamos la lectura de los datos de interes
df, meta = pyreadstat.read_sas7bdat('../../data/raw/CY08MSP_STU_QQQ.SAS7BDAT', metadataonly=True)

In [4]:
print(f'Numero de filas: {meta.number_rows} \nNumero de columnas: {meta.number_columns}')

Numero de filas: 613744 
Numero de columnas: 1278


Observamos la informacion contenida en las columnas (Solo los primeros datos)

In [5]:
#Pasamos el dicionario a una lista e iteramos sobre lso valores de la lista
first_values = list(meta.column_names_to_labels.items())[:10]
for clave,valor in first_values:
    print(f"{clave}: {valor}")

CNT: Country code 3-character
CNTRYID: Country Identifier
CNTSCHID: Intl. School ID
CNTSTUID: Intl. Student ID
CYC: PISA Assessment Cycle (2 digits + 2 character Assessment type - MS/FT)
NatCen: National Centre 6-digit Code
STRATUM: Stratum ID 5-character (cnt + original stratum ID)
SUBNATIO: Adjudicated sub-region code 7-digit code (3-digit country code + region ID + stratum ID)
REGION: REGION
OECD: OECD country


# Datos resultados lectura PISA

In [6]:
columnas_lectura = [col for col in meta.column_names if 'PV' and 'READ' in col]


In [7]:
print(f"Columnas encontradas: {len(columnas_lectura)}")
print(columnas_lectura)

Columnas encontradas: 10
['PV1READ', 'PV2READ', 'PV3READ', 'PV4READ', 'PV5READ', 'PV6READ', 'PV7READ', 'PV8READ', 'PV9READ', 'PV10READ']


In [8]:
#Añadimos el resto de columnas de interes para realizar el estudio : 
columnas_interes = ['CNT', 'W_FSTUWT'] + columnas_lectura

In [9]:
df_pisa, meta_real = pyreadstat.read_sas7bdat(
    '../../data/raw/CY08MSP_STU_QQQ.SAS7BDAT', 
    usecols=columnas_interes
)

In [10]:
df_pisa.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 12 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   CNT       613744 non-null  str    
 1   W_FSTUWT  613744 non-null  float64
 2   PV1READ   613744 non-null  float64
 3   PV2READ   613744 non-null  float64
 4   PV3READ   613744 non-null  float64
 5   PV4READ   613744 non-null  float64
 6   PV5READ   613744 non-null  float64
 7   PV6READ   613744 non-null  float64
 8   PV7READ   613744 non-null  float64
 9   PV8READ   613744 non-null  float64
 10  PV9READ   613744 non-null  float64
 11  PV10READ  613744 non-null  float64
dtypes: float64(11), str(1)
memory usage: 57.9 MB


In [11]:
display(df_pisa.head())
display(df_pisa.tail())

,CNT,W_FSTUWT,PV1READ,PV2READ,PV3READ,PV4READ,PV5READ,PV6READ,PV7READ,PV8READ,PV9READ,PV10READ
0,ALB,3.15874,247.571,322.919,220.679,220.125,259.780,242.733,253.962,299.988,164.397,265.872
1,ALB,4.34524,258.472,305.879,328.252,235.438,292.872,361.678,294.129,213.902,267.050,331.327
2,ALB,7.83860,284.670,386.451,329.009,318.836,269.201,333.165,314.457,290.453,269.663,321.880
3,ALB,8.49148,321.547,256.187,235.363,362.717,310.844,259.632,352.668,334.544,252.265,321.986
4,ALB,3.70957,464.366,455.299,496.037,510.626,494.031,430.135,486.110,478.975,471.980,579.130


,CNT,W_FSTUWT,PV1READ,PV2READ,PV3READ,PV4READ,PV5READ,PV6READ,PV7READ,PV8READ,PV9READ,PV10READ
613739,UZB,69.40366,315.903,276.158,297.158,325.051,313.258,321.272,318.046,315.303,321.744,321.813
613740,UZB,63.18910,278.035,261.609,294.946,241.376,247.087,307.526,254.580,276.609,247.353,310.424
613741,UZB,65.47339,310.081,309.707,323.988,347.815,332.785,314.931,351.789,379.927,315.838,317.512
613742,UZB,66.15595,388.350,380.237,361.197,332.856,341.325,334.191,353.207,357.694,346.518,334.924
613743,UZB,65.47339,340.671,394.842,385.103,390.448,344.446,371.540,401.644,323.263,449.953,431.542


In [12]:
rendimiento_lectura_PISA= consolidar_puntuaciones_pisa(df_pisa,competencia='READ')

In [13]:
#Este es el dataframe que contiene cada media de los PV de lectura independiente
rendimiento_lectura_PISA.head()

,codigo_pais,media_read_pisa,varianza_imputacion_read,pv1_read,pv2_read,pv3_read,pv4_read,pv5_read,pv6_read,pv7_read,pv8_read,pv9_read,pv10_read
0,ALB,358.434758,0.243473,358.446138,357.929069,357.651371,359.325486,358.970893,358.305053,358.119524,358.307358,358.743572,358.549119
1,ARE,417.349750,0.470194,417.865054,417.089223,418.224166,417.041867,417.314945,416.784893,416.954522,416.524781,417.017356,418.680691
2,ARG,400.744043,0.290839,400.923353,400.699265,400.242856,401.074315,401.545441,400.016070,401.437064,400.716887,399.990788,400.794395
3,AUS,498.050940,0.404705,498.207710,498.848211,496.968283,498.834798,497.360231,498.315057,497.741116,497.481461,498.284781,498.467749
4,AUT,480.405847,0.482573,480.604566,480.834388,480.355423,479.825819,480.231034,481.730074,479.132398,480.782608,480.599008,479.963153


In [14]:
rendimiento_lectura_PISA.shape

(80, 13)

In [15]:
lectura_top_cnt=rendimiento_lectura_PISA.sort_values(by='media_read_pisa',ascending=False)

In [16]:
fig=px.bar(lectura_top_cnt,y='media_read_pisa',x='codigo_pais')

In [17]:
fig.show()

In [ ]:
ruta_salida = '../../data/intermediate/pisa_lectura_consolidado.parquet'
rendimiento_lectura_PISA.to_parquet(ruta_salida, index=False)